In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully!")

# diagnostik
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

# tampilan
sns.set_style("whitegrid")
%matplotlib inline

print("Libraries loaded successfully!")


In [ ]:
# =========================
# Imports
# =========================
import numpy as np
import pandas as pd
import statsmodels.api as sm

# =========================
# Data generation
# =========================
np.random.seed(42)

n = 300  # number of households

df = pd.DataFrame({
    "household_id": range(1, n + 1),
    "village": np.random.choice(["Village A", "Village B", "Village C"], n),
    "monthly_income": np.random.normal(1_500_000, 500_000, n).clip(300_000),
    "house_condition": np.random.choice(
        ["poor", "average", "good"], n, p=[0.45, 0.35, 0.20]
    ),
    "num_dependents": np.random.randint(0, 6, n)
})

# =========================
# Eligibility (ground truth)
# =========================
df["eligible_actual"] = (
    (df["monthly_income"] < 1_200_000) &
    (df["house_condition"] == "poor") &
    (df["num_dependents"] >= 2)
).astype(int)

# =========================
# Logistic Regression
# =========================
X = df[["monthly_income", "num_dependents"]]
X = sm.add_constant(X)

y = df["eligible_actual"]

logit_model = sm.Logit(y, X).fit(disp=False)

print(logit_model.summary())

# =========================
# Simulate targeting errors
# =========================
df["received_assistance"] = np.where(
    df["eligible_actual"] == 1,
    np.random.choice([1, 0], n, p=[0.7, 0.3]),   # exclusion error
    np.random.choice([0, 1], n, p=[0.85, 0.15])  # inclusion error
)

df["targeting_status"] = np.select(
    [
        (df["eligible_actual"] == 1) & (df["received_assistance"] == 1),
        (df["eligible_actual"] == 1) & (df["received_assistance"] == 0),
        (df["eligible_actual"] == 0) & (df["received_assistance"] == 1)
    ],
    ["accurate", "exclusion_error", "inclusion_error"],
    default="accurate"
)

print(df.head())
print("\nTargeting summary:")
print(df["targeting_status"].value_counts())


In [ ]:
# 1 = mis-targeted, 0 = correctly targeted
df["mis_targeted"] = df["targeting_status"].apply(
    lambda x: 1 if x in ["inclusion_error", "exclusion_error"] else 0
)

df["mis_targeted"].value_counts()


In [ ]:
df.groupby("mis_targeted")[["monthly_income", "num_dependents"]].mean()
pd.crosstab(df["house_condition"], df["mis_targeted"], normalize="index")



In [ ]:
import statsmodels.api as sm

X = df[["monthly_income", "num_dependents"]]
X = sm.add_constant(X)

y = df["mis_targeted"]

logit_model = sm.Logit(y, X).fit()
logit_model.summary()


## Logistic Regression Interpretation

The dependent variable is **mis_targeted**, indicating whether a household
experienced mistargeting in social assistance distribution.

Key findings:
- **Monthly income** has a negative coefficient, suggesting higher income
  households tend to have lower probability of mistargeting, although the
  effect is not statistically significant.
- **Number of dependents** has a positive coefficient, indicating households
  with more dependents are more likely to experience mistargeting.
- The model converges successfully, indicating stable estimation.
